<img src="logo.png" alt="Vegeta" width="240">

# Designing a full quadcopter frame — CAD → FEA → CFD → print, with a revision history

A 250 mm-class (5-inch) X-frame, printed in one piece. This notebook is the *whole* Vegeta
workflow on one product: mission and mass budget → parametric CAD (Dedalus) → two structural load
cases (Talos) → design revisions compared in a `vegeta.core` workspace → forward-flight drag with a
canopy (Aeromant) → slicing (Mellonia).

Rules of the house: every number below (loads, material, thrust) is an **explicit engineering input
written in this notebook**, not something Vegeta guessed. Nothing runs unless you run the cell;
every result is a plain file under `_runs/quad/`. Values are illustrative — replace them with yours.

In [ ]:
import math, shutil
from pathlib import Path
import numpy as np
import pandas as pd
from vegeta import dedalus, talos, aeromant, mellonia, core
from vegeta.dedalus import viz as dviz
from vegeta.talos import viz as tviz
from vegeta.aeromant import viz as aviz
from vegeta.mellonia import viz as mviz
from vegeta.mellonia.examples import GENERIC_PLA_0_2MM
from tqdm.auto import tqdm

RUNS = Path("_runs/quad"); shutil.rmtree(RUNS, ignore_errors=True); RUNS.mkdir(parents=True)

## 1. Mission and mass budget

A freestyle/utility quad: 4S battery, 5-inch props, all-up weight (AUW) below ~500 g, thrust-to-weight
≥ 4 at full throttle. Component masses are datasheet-style values you would take from the parts you
actually order; the frame mass is filled in later from the CAD volume.

In [ ]:
parts = pd.DataFrame([
    ("motors 2306 (4x)",        4 * 30.0),
    ("propellers 5x4.3 (4x)",   4 * 5.0),
    ("4-in-1 ESC",              15.0),
    ("flight controller",       10.0),
    ("battery 4S 1500 mAh",     180.0),
    ("camera + VTX + antenna",  35.0),
    ("receiver, wiring, bolts", 30.0),
], columns=["part", "mass_g"]).set_index("part")

MAX_THRUST_PER_MOTOR_N = 8.0      # ~815 gf per motor at full throttle on 4S with a 5x4.3 prop (datasheet-style value)
G = 9.81
parts

## 2. Parametric CAD (Dedalus)

The frame is one fused solid: a round centre plate with the flight-controller stack holes (30.5 mm),
four tapered arms at 45°, and round motor pads with a 16×16 mm M3 pattern. Loads and supports will
attach to the **bolt-hole cylinders** — the faces a real motor or stack actually pulls on — so no
special "load face" is needed in the CAD. An optional elliptic canopy encloses the electronics for the CFD.

A design note from the first attempt: with 26 mm pads the M3 holes left a 0.1 mm wall at the pad edge —
CadQuery builds it happily, Gmsh cannot mesh it. The design now checks the wall thickness itself.

The design lives in its own file so it can be revised, diffed and (notebook 07) handed to the AI copilot.

In [ ]:
design_file = RUNS / "quad.py"
design_file.write_text('''import math
import cadquery as cq
from vegeta.dedalus import Design, Parameter


class QuadFrame(Design):
    """X-frame quadcopter: round centre plate, four tapered arms, round motor pads. One printable solid.
    Motor and stack bolt holes are through holes; loads and supports attach to their cylindrical faces."""

    parameters = [
        Parameter("wheelbase", 250.0, "mm", min=100, description="motor-to-motor diagonal"),
        Parameter("arm_width", 12.0, "mm", min=4, description="arm width at the root"),
        Parameter("arm_height", 6.0, "mm", min=2, description="arm thickness (Z)"),
        Parameter("taper", 0.7, "", min=0.3, max=1.0, description="arm width at the pad / at the root"),
        Parameter("plate_size", 70.0, "mm", min=30, description="centre plate diameter"),
        Parameter("plate_thickness", 6.0, "mm", min=1, description="centre plate thickness (>= arm_height: the arms cross inside it)"),
        Parameter("pad_diameter", 30.0, "mm", min=10, description="motor pad diameter"),
        Parameter("motor_pattern", 16.0, "mm", min=5, description="motor bolt square (M3, 16x16)"),
        Parameter("motor_hole", 3.2, "mm", min=1),
        Parameter("stack_pattern", 30.5, "mm", min=10, description="flight-controller stack bolt square"),
        Parameter("stack_hole", 3.2, "mm", min=1),
        Parameter("canopy_height", 0.0, "mm", min=0, description="elliptic dome over the plate (0 = none)"),
    ]

    def build(self, p):
        R = p["wheelbase"] / 2
        w, h, t = p["arm_width"], p["arm_height"], p["plate_thickness"]
        if p["pad_diameter"] < p["motor_pattern"] * math.sqrt(2) + p["motor_hole"] + 3.0:
            raise ValueError("pad_diameter leaves less than 1.5 mm of wall around the motor bolt holes")
        if t < h:
            raise ValueError("plate_thickness must be >= arm_height (the four arms cross inside the plate)")
        s = p["plate_size"]
        frame = cq.Workplane("XY").circle(s / 2).extrude(t)
        pads = cq.Workplane("XY")
        holes = cq.Workplane("XY")
        stack = p["stack_pattern"] / 2
        holes = holes.pushPoints([(stack, stack), (-stack, stack), (stack, -stack), (-stack, -stack)]) \
            .circle(p["stack_hole"] / 2).extrude(50)
        for k in range(4):
            ang = math.radians(45 + 90 * k)
            cx, cy = R * math.cos(ang), R * math.sin(ang)
            arm = (cq.Workplane("XY")
                   .polyline([(0, -w / 2), (R, -w * p["taper"] / 2), (R, w * p["taper"] / 2), (0, w / 2)])
                   .close().extrude(h).rotate((0, 0, 0), (0, 0, 1), math.degrees(ang)))
            pad = cq.Workplane("XY").center(cx, cy).circle(p["pad_diameter"] / 2).extrude(h)
            frame = frame.union(arm).union(pad)
            m = p["motor_pattern"] / 2
            holes = holes.pushPoints([(cx + m, cy + m), (cx - m, cy + m), (cx + m, cy - m), (cx - m, cy - m)]) \
                .circle(p["motor_hole"] / 2).extrude(50)
        if p["canopy_height"] > 0:
            dome = (cq.Workplane("XY").workplane(offset=t)
                    .ellipse(s / 2 - 6, s / 2 - 6).extrude(p["canopy_height"])
                    .faces(">Z").edges().fillet(p["canopy_height"] * 0.9))
            frame = frame.union(dome)
        return frame.cut(holes)
''')
frame_design = dedalus.load_design(f"{design_file}:QuadFrame")
pd.DataFrame(frame_design.params.table()).set_index("name")

In [ ]:
frame = frame_design.generate()
frame          # interactive CadQuery view

In [ ]:
FRAME_DENSITY_G_MM3 = 1.25e-3     # PETG-CF, nominal (1.25 g/cm^3)
frame_mass_g = frame.volume * FRAME_DENSITY_G_MM3
parts.loc["frame (printed, from CAD volume)"] = round(frame_mass_g, 1)
AUW_g = parts["mass_g"].sum()
hover_thrust_per_motor_N = AUW_g / 1000 * G / 4
print(f"frame {frame_mass_g:.1f} g  |  AUW {AUW_g:.0f} g  |  hover thrust/motor {hover_thrust_per_motor_N:.2f} N  "
      f"|  thrust-to-weight {4 * MAX_THRUST_PER_MOTOR_N / (AUW_g / 1000 * G):.2f}")
parts

In [ ]:
dviz.show(dviz.plot3d(frame, show_edges=False))

In [ ]:
p0 = frame_design.resolve()
R = p0["wheelbase"] / 2
cx = R * math.cos(math.radians(45))
fig = dviz.plot_sections(frame, normal="x", positions=[0.0, 40.0, cx], cols=3)      # plate, arm, motor pad
fig = dviz.plot_sections(frame, normal="z", positions=[1.0, 3.0, 5.0], cols=3)      # in-plane cuts

## 3. Structural load cases (Talos)

Two explicit load cases, both with the frame bolted to the FC stack (the four stack-hole cylinders are
fixed); thrust enters through the four motor bolt holes of each pad:

| case | load | what it represents |
|---|---|---|
| `max_thrust` | +8 N up on each motor's bolt holes | full-throttle punch-out, symmetric |
| `hard_landing` | +40 N up on **one** motor's bolt holes | landing on one arm — a *chosen* design load, 5× the motor's max thrust |

Material: PETG-CF, nominal values (E = 4.8 GPa, ν = 0.38, yield 45 MPa). Printed parts are anisotropic;
these numbers are a starting point for comparison between revisions, not a certification.

Regions are selected geometrically from the parameters (boxes around each bolt pattern),
so the same factory works for every revision. `talos.inspect_step` shows the surface list if you want
to check what was selected.

In [ ]:
PETG_CF = talos.Material("PETG-CF", youngs_modulus=4800.0, poissons_ratio=0.38, density=1.25e-9,
                         yield_strength=45.0, source="nominal filament datasheet values, XY orientation")

def frame_regions(p):
    """Bolt-hole cylinders of each motor (motor0..3) and of the stack (stack0..3), by bounding box."""
    R, m, r = p["wheelbase"] / 2, p["motor_pattern"] / 2, p["motor_hole"] / 2 + 0.2
    regions = []
    for k in range(4):
        a = math.radians(45 + 90 * k)
        cx, cy = R * math.cos(a), R * math.sin(a)
        regions.append(talos.SurfacesInBox(f"motor{k}", (cx - m - r, cy - m - r, -0.1, cx + m + r, cy + m + r, 100.0)))
    s, r = p["stack_pattern"] / 2, p["stack_hole"] / 2 + 0.2
    for k, (sx, sy) in enumerate([(s, s), (-s, s), (s, -s), (-s, -s)]):
        regions.append(talos.SurfacesInBox(f"stack{k}", (sx - r, sy - r, -0.1, sx + r, sy + r, 100.0)))
    return regions

def frame_model(step, p, loads, name):
    return talos.StructuralModel(step, "mm-N-MPa", PETG_CF, frame_regions(p),
                                 supports=[talos.FixedSupport(f"stack{k}") for k in range(4)], loads=loads,
                                 mesh_settings=talos.MeshSettings(element_size=4.0), name=name)

def max_thrust(rev):
    return frame_model(rev.step, rev.params, [talos.Force(f"motor{k}", fz=MAX_THRUST_PER_MOTOR_N) for k in range(4)], "max_thrust")

def hard_landing(rev):
    return frame_model(rev.step, rev.params, [talos.Force("motor0", fz=40.0)], "hard_landing")

In [ ]:
frame.export_step(RUNS / "preview.step")
talos.inspect_step(RUNS / "preview.step", units="mm-N-MPa")

## 4. A workspace with revisions (vegeta.core)

Every revision is immutable: parameters, source hash, geometry and evaluations are recorded once.
Revision r1 is the baseline; we will branch from it.

In [ ]:
ws = core.Workspace.create(RUNS / "workspace", name="quad frame study")
quad = ws.add_design("frame", f"{design_file}:QuadFrame")
r1 = quad.new_revision(note="baseline: 12x6 arms, taper 0.7")
r1.generate()
ws.status()

In [ ]:
ev_t = r1.run_fea("max_thrust", max_thrust, progress=True)
ev_l = r1.run_fea("hard_landing", hard_landing, progress=True)
ws.status()

In [ ]:
mesh_res, solve_res = ev_t.tool_results[0], ev_t.tool_results[-1]
tviz.show(tviz.plot_problem(max_thrust(r1), mesh_res))

In [ ]:
if ev_t.ok:
    tviz.show(tviz.plot_results(solve_res, field="von_mises"))

In [ ]:
if ev_t.ok:
    fig = tviz.plot_section(solve_res, normal="z", origin=(0, 0, 3.0), field="von_mises")   # mid-thickness map
    fig = talos.plot_deformed(solve_res)

In [ ]:
if ev_l.ok:
    tviz.show(tviz.plot_results(ev_l.tool_results[-1], field="|U|"))
    print(ev_l)

### Reading the numbers
`max_displacement` is the tip deflection of a pad; `safety_factor_yield` is yield / peak nodal von Mises.
The peak sits at a bolt hole (support/load singularity) — compare it *between revisions*, do not read it
as an absolute. Reactions are returned per stack hole; CalculiX's reaction totals exclude loads on
supported nodes (none here).

## 5. Iterate: three candidate frames

The baseline arms are floppy under a hard landing. Two classic fixes, as branches of r1:
taller arms (stiffness ∝ h³) and wider, less tapered arms (more mass, more damage tolerance).
Every branch is generated and analysed with **exactly the same factories** — that is the point of
recording the load cases as code.

In [ ]:
def branch_once(parent, note, **params):
    # re-running this cell reuses the revision instead of creating a new one each time
    return next((r for r in ws.revisions() if r.record.get("note") == note), None) or parent.branch(note=note, **params)

r2 = branch_once(r1, "taller arms", arm_height=8.0, plate_thickness=8.0)
r3 = branch_once(r1, "wider arms, stronger taper", arm_width=16.0, taper=0.55)

# 4 FEA jobs (~25-35k second-order tets each): about a minute on a laptop, longer on a slow one.
# Safe to interrupt and re-run: finished work is skipped, an interrupted evaluation is moved aside.
jobs = [(r, case) for r in (r2, r3) for case in (max_thrust, hard_landing)]
for r, case in tqdm(jobs, desc="revisions x load cases"):
    if not r.is_generated:
        r.generate()
    if r.evaluation("fea", case.__name__) is None:
        r.run_fea(case.__name__, case, progress=True)
ws.status()

In [ ]:
def row(rev):
    vol = rev.geometry_summary()["metrics"]["volume"]
    d = {"rev": rev.id, "note": rev.record.get("note", ""), "arm_h": rev.params["arm_height"],
         "arm_w": rev.params["arm_width"], "taper": rev.params["taper"], "mass_g": vol * FRAME_DENSITY_G_MM3}
    for case in ("max_thrust", "hard_landing"):
        ev = rev.evaluation("fea", case)
        d[f"{case}_defl_mm"] = ev.metrics.get("max_displacement", np.nan) if ev and ev.ok else np.nan
        d[f"{case}_SF"] = ev.metrics.get("safety_factor_yield", np.nan) if ev and ev.ok else np.nan
    return d

table = pd.DataFrame([row(r) for r in (r1, r2, r3)]).set_index("rev").round(2)
table

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
table.plot.bar(y=["max_thrust_defl_mm", "hard_landing_defl_mm"], ax=ax[0], title="pad deflection [mm]")
ax[1].scatter(table["mass_g"], table["hard_landing_SF"], s=80)
for rid, r in table.iterrows():
    ax[1].annotate(rid, (r["mass_g"], r["hard_landing_SF"]), textcoords="offset points", xytext=(6, 4))
ax[1].set(xlabel="frame mass [g]", ylabel="SF, hard landing", title="mass vs safety factor"); ax[1].grid(alpha=0.3)
fig.tight_layout()

In [ ]:
best = table["hard_landing_SF"].idxmax()
preferred = ws.revision(best)
preferred.label("preferred", note=f"best hard-landing SF ({table.loc[best, 'hard_landing_SF']}) at {table.loc[best, 'mass_g']} g")
for r in (r1, r2, r3):
    if r.id != best:
        r.label("rejected", note="lower safety factor than the preferred revision")
ws.status()

## 6. Forward flight drag with a canopy (Aeromant)

The preferred frame gets an elliptic canopy over the electronics. We run steady RANS (k-ω SST) at
15 m/s with the frame at 0° pitch (a real quad flies nose-down; that is a parameter for the next
revision), coarse mesh for a *comparative* drag figure. Reference area = plate footprint.

In [ ]:
r4 = preferred.branch(canopy_height=25.0, note="canopy for CFD")
r4.generate(stl_tolerance=0.1)
cases = {}

def forward_flight(rev, workdir):
    p = rev.params
    case = aeromant.CFDCase(
        "rans_ksst_external", rev.stl,
        dict(velocity=15.0, kinematic_viscosity=1.5e-5, density=1.2,
             reference_area=(p["plate_size"] / 1000) ** 2, reference_length=0.1, center_of_rotation=(0, 0, 0),
             iterations=250, surface_level=3, near_level=2, wake_level=1, cells_per_length=2.0),
        workdir=workdir, geometry_units="mm", environment=aeromant.OpenFOAMEnvironment.detect())
    cases[rev.id] = case
    return case

ev_cfd = r4.run_cfd("forward_15ms", forward_flight, progress=True)
print(ev_cfd)

In [ ]:
case = cases[r4.id]
aviz.show(aviz.plot_setup(case))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_mesh_slice(case, normal="y"))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_field_slice(case, "U", normal="y"))
    fig = aviz.plot_section(case, "p", normal="y", zoom=2)

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_streamlines(case))

In [ ]:
if ev_cfd.ok:
    aviz.show(aviz.plot_surface_pressure(case))
    F = ev_cfd.metrics["drag_force_N"]
    print(f"Cd {ev_cfd.metrics['Cd']:.3f}  drag {F:.2f} N at 15 m/s  ->  parasite power {F * 15:.1f} W "
          f"(rotor induced power dominates; this is the airframe share)")

## 7. Print the preferred frame (Mellonia)

Flat on the bed, no supports needed. The example profile is generic PLA — swap in
your printer's exported PETG-CF profile with `PrintSettings.from_ini(...)`.

In [ ]:
ev_p = preferred.run_print("flat", GENERIC_PLA_0_2MM, mellonia.Orientation())
print(ev_p)

In [ ]:
if ev_p.ok:
    prn = ev_p.tool_results[0]
    mviz.show(mviz.plot_toolpath(prn))

In [ ]:
if ev_p.ok:
    fig = mviz.plot_layer_grid(prn, n=6, cols=3)

## 8. Where we are

The workspace is the record: each revision folder holds `revision.json`, the STEP/STL, and one
directory per evaluation with the tools' native files (`mesh.msh`, `.inp`, `.frd`, the OpenFOAM case,
the G-code) plus every command that was run.

In [ ]:
parts.loc["frame (printed, from CAD volume)"] = round(table.loc[best, "mass_g"], 1)
print(f"AUW {parts['mass_g'].sum():.0f} g with the preferred frame ({best})")
ws.status()

In [ ]:
for p in sorted((RUNS / "workspace" / "revisions" / best).rglob("*"))[:30]:
    print(p.relative_to(RUNS / "workspace"))

**Next steps an engineer would take:** pitch the frame in the CFD (`rotate` the STL or add a `pitch`
parameter), add the landing-gear and arm-break-away features, run the hard-landing case with the
printed-part anisotropy (a second material with lower Z properties), and let the AI copilot
(notebook 07) propose weight-saving cut-outs on `_runs/quad/quad.py` — every proposal is built and
measured before you accept it.